# model-train-eval-toggle-around-sample — faded example 3: Fill the finally restore in eval_mode

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `model-train-eval-toggle-around-sample`. Running the beacon reports progress on the `GAN: model.train/eval toggle around sample` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: model.train/eval toggle around sample` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`model-train-eval-toggle-around-sample`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "model-train-eval-toggle-around-sample"
DD_SUBTOPIC = "GAN: model.train/eval toggle around sample"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

An exception-safe eval context manager restores the prior training state in a `finally` block so it runs even when the body raises. It only calls `model.train()` if the model was training on entry.

## Faded exercise 3

Complete the `eval_mode(model)` context manager. The capture, eval switch, and no_grad yield are given; fill in the finally clause that conditionally restores training mode.

**Fill in:** the finally clause restoring train mode only if it was training before

In [ ]:
import torch as t
import torch.nn as nn
import contextlib

@contextlib.contextmanager
def eval_mode(model):
    was_training = model.training
    model.eval()
    try:
        with t.no_grad():
            yield model
    finally:
        raise NotImplementedError()  # TODO: the finally clause restoring train mode only if it was training before


def _test():
    t.manual_seed(53)
    # train-start: restored to train even after an exception
    m = nn.Linear(2, 2).train()
    hit = False
    try:
        with eval_mode(m):
            assert not m.training
            raise ValueError('x')
    except ValueError:
        hit = True
    assert hit
    assert m.training is True, 'must restore train after exception'
    # eval-start: stays in eval
    m2 = nn.Linear(2, 2).eval()
    with eval_mode(m2):
        pass
    assert m2.training is False, 'eval-start model must stay in eval'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn
import contextlib

@contextlib.contextmanager
def eval_mode(model):
    was_training = model.training
    model.eval()
    try:
        with t.no_grad():
            yield model
    finally:
        if was_training:
            model.train()
```
</details>